In [ ]:
!pip install --upgrade google-cloud-bigquery

In [ ]:
pip install --upgrade pip

In [ ]:
#!pip install db-dtypes pandas-gbq
#!pip install db-dtypes
#!pip install google-cloud-bigquery-storage pyarrow

from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project= 'moonlit-parsec-485600-p0')

# This query stacks the tables from different dates and unnest the stacked tables
sql = """
SELECT 
    fullVisitorId,
    date,
    p.v2ProductName, 
    p.v2ProductCategory
FROM
    `bigquery-public-data.google_analytics_sample.ga_sessions_*`,
    UNNEST(hits) AS h,
    UNNEST(h.product) AS p
WHERE
    _TABLE_SUFFIX BETWEEN '20170101' AND '20170630'
    AND h.transaction.transactionId is NOT NULL
"""

df = client.query(sql).to_dataframe()
print(f"Successfully pulled {len(df):,} rows!")
df.head()


In [ ]:
df = df.rename(columns = {
    'fullVisitorId' : 'user_id',
    'v2ProductName': 'product_name',
    'v2ProductCategory': 'product_category'
})

#df.info()

agg_df = df.groupby(['user_id','product_name']).size().reset_index(name = 'interaction_count')
agg_df

In [ ]:
# How many products does the average user interact with?
user_activity = agg_df.groupby('user_id').size()

print(f"Average items per user: {user_activity.mean():.2f}")
print(f"Max items by a single user: {user_activity.max()}")
print (user_activity.sort_values(ascending = False).head(10))

# Visualize
user_activity.sort_values(ascending = False).head(10).plot(
    kind = 'bar',
    color = 'skyblue',
    edgecolor = 'black'
)

plt.title('Top 10 Most Active Users')
plt.ylabel('Number of Unique Products')
plt.xlabel('User ID (Truncated)')
plt.xticks(rotation = 45)
plt.show()

In [ ]:
# What are the top 10 most popular products?
top_products = agg_df.groupby('product_name')['interaction_count'].sum().sort_values(ascending=False)
print(top_products.head(10))

top_10_df = top_products.head(10).reset_index()

plt.figure(figsize = (12,6))
sns.set_theme(style = "whitegrid")

ax = sns.barplot(
    data = top_10_df,
    x = 'interaction_count',
    y = 'product_name',
    legend = False,
    palette = 'viridis'
)

for container in ax.containers:
    ax.bar_label(container, fmt='%g', padding=5, fontsize=11, fontweight='bold')

plt.title('Top 10 Most Popular Products', fontsize=16)
plt.xlabel('Total Interactions', fontsize=12)
plt.ylabel('Product Name', fontsize=12)
plt.xlim(0, top_10_df['interaction_count'].max() * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
#product_selection = df.groupby('product_name')['unit_price'].first().reset_index()

#product_selection = df.groupby('product_name')['unit_price'].mean().reset_index()

# Compared the mean and first values, noticed a difference thus, decided to use unique to get all unique value
# This looks at only these two columns and removes any rows that are identical
product_selection = df[['product_name', 'unit_price']].drop_duplicates().reset_index(drop=True)

# Sort it by name so you can see the two Sweatshirt prices together
product_selection = product_selection.sort_values('product_name')

#print(product_selection)

# Take prices of only top 10 items
subset_product = product_selection[product_selection['product_name'].isin(top_10_names)]
subset_product




In [ ]:
# Checking to see if prices are different due to inflation or due to variety
# Pick a few products that had multiple price points to investigate
products_with_multiple_prices = (
    df.groupby('product_name')['unit_price']
    .nunique()
    .reset_index()
    .query('unit_price > 1')
    .sort_values('unit_price', ascending=False)
    .head(6)['product_name']
    .tolist()
)

plot_df = df[df['product_name'].isin(products_with_multiple_prices)].copy()
plot_df['date'] = pd.to_datetime(plot_df['date'], format='%Y%m%d')

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, product in enumerate(products_with_multiple_prices):
    product_df = plot_df[plot_df['product_name'] == product].sort_values('date')
    axes[i].scatter(product_df['date'], product_df['unit_price'], alpha=0.6, s=25, color='steelblue')
    axes[i].set_title(product[:40], fontsize=9)
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('Unit Price')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Exact Price Points Over Time', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Only filter out zero or negative prices
df_clean = df[df['unit_price'] > 0].copy()

# Check what got removed
removed = df[df['unit_price'] <= 0]
print(f"Rows removed: {len(removed)}")
print(f"Rows retained: {len(df_clean)}")

if len(removed) > 0:
    print("\nRemoved products:")
    print(removed[['product_name', 'unit_price', 'date']].head(10))

In [ ]:
df_clean['date'] = pd.to_datetime(df_clean['date'], format='%Y%m%d')

most_recent_price = (
    df_clean.sort_values('date')
            .groupby('product_name')
            .last()['unit_price']
            .reset_index()
            .rename(columns={'unit_price': 'current_price'})
)

print(f"\nUnique products with valid prices: {len(most_recent_price)}")
print(most_recent_price.sort_values('current_price').head(10))

In [ ]:
# Filter to top 10 products by interaction score
df_clean['date'] = pd.to_datetime(df_clean['date'], format='%Y%m%d')

product_scores = (
    df_clean.groupby('product_name')
    .agg(purchase_count=('unit_price', 'count'),
         total_spend=('unit_price', 'sum'))
    .assign(interaction_score=lambda x: x['purchase_count'] * np.log1p(x['total_spend']))
    .sort_values('interaction_score', ascending=False)
)

top_10_names = product_scores.head(10).index.tolist()
df_top10 = df_clean[df_clean['product_name'].isin(top_10_names)]

# Build co-purchase matrix
co_matrix = pd.DataFrame(0.0, index=top_10_names, columns=top_10_names)

visitor_products = df_top10.groupby('user_id')['product_name'].apply(set)

from itertools import combinations

for products in visitor_products:
    products = list(products & set(top_10_names))
    if len(products) < 2:
        continue
    for p1, p2 in combinations(products, 2):
        score = (product_scores.loc[p1, 'interaction_score'] *
                 product_scores.loc[p2, 'interaction_score'])
        co_matrix.loc[p1, p2] += score
        co_matrix.loc[p2, p1] += score

# Plot
short_names = [name[:30] + '...' if len(name) > 30 else name for name in top_10_names]
co_matrix.index   = short_names
co_matrix.columns = short_names

plt.figure(figsize=(12, 10))
sns.heatmap(
    co_matrix,
    annot=True,
    fmt='.2e',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='white',
    square=True,
    cbar_kws={'label': 'Co-purchase Interaction Score'}
)

plt.title('Top 10 Products — Co-Purchase Interaction Heatmap',
          fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

# use iloc positions to avoid duplicate label issues
pairs = []
for i, p1 in enumerate(short_names):
    for j, p2 in enumerate(short_names):
        if j <= i:  # upper triangle only, skip diagonal
            continue
        score_val = co_matrix.iloc[i, j]  # positional, always returns a scalar
        pairs.append((p1, p2, score_val))

pairs_df = pd.DataFrame(pairs, columns=['Product A', 'Product B', 'Score'])
print("Top co-purchased pairs:")
print(pairs_df.sort_values('Score', ascending=False).head(5).to_string(index=False))

In [ ]:
from itertools import combinations

# Get top 10 products by interaction score
df_clean['date'] = pd.to_datetime(df_clean['date'], format='%Y%m%d')

product_scores = (
    df_clean.groupby('product_name')
    .agg(purchase_count=('unit_price', 'count'),
         total_spend=('unit_price', 'sum'))
    .assign(interaction_score=lambda x: x['purchase_count'] * np.log1p(x['total_spend']))
    .sort_values('interaction_score', ascending=False)
)

top_10_names = product_scores.head(10).index.tolist()
df_top10 = df_clean[df_clean['product_name'].isin(top_10_names)]

# Build user-item matrix then correlate
# Pivot: rows = users, columns = products, values = interaction count
user_item_matrix = (
    df_top10.groupby(['user_id', 'product_name'])
    .size()
    .unstack(fill_value=0)
)

# Correlate across products (columns) — this is what your second heatmap did
corr_matrix = user_item_matrix.corr()

# Apply label map for duplicates + truncate long names
label_map = {
    "Google Men's 100% Cotton Short Sleeve Hero Tee White": "Cotton Tee (White)",
    "Google Men's 100% Cotton Short Sleeve Hero Tee Black": "Cotton Tee (Black)"
}

short_names = [
    label_map.get(name, name[:30] + '...' if len(name) > 30 else name)
    for name in corr_matrix.columns
]

corr_matrix.index   = short_names
corr_matrix.columns = short_names

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',           # correlation is -1 to 1, no need for scientific notation
    cmap='YlGnBu',
    linewidths=0.5,
    linecolor='white',
    square=True,
    vmin=-1, vmax=1,     # fix scale so colours are meaningful
    cbar_kws={'label': 'Pearson Correlation'}
)

plt.title('Top 10 Products — Co-occurrence Correlation Heatmap',
          fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

# Top correlated pairs
pairs = []
for i, p1 in enumerate(short_names):
    for j, p2 in enumerate(short_names):
        if j <= i:  # upper triangle only, skip diagonal
            continue
        pairs.append((p1, p2, corr_matrix.iloc[i, j]))

pairs_df = pd.DataFrame(pairs, columns=['Product A', 'Product B', 'Correlation'])
print("Top correlated pairs:")
print(pairs_df.sort_values('Correlation', ascending=False).head(5).to_string(index=False))

print("\nLeast correlated pairs (avoid recommending together):")
print(pairs_df.sort_values('Correlation').head(5).to_string(index=False))